[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bobleesj/quantem.widget/blob/main/notebooks/bin2d/bin2d_simple.ipynb)
# Bin2D — Quick Demo
Interactive 2D image binning with side-by-side original/binned comparison.

In [28]:
# Install in Google Colab
try:
    import google.colab
    !pip install -q -i https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ quantem-widget
except ImportError:
    pass  # Not in Colab, skip

In [29]:
try:
    %load_ext autoreload
    %autoreload 2
    %env ANYWIDGET_HMR=1
except Exception:
    pass  # autoreload unavailable (Colab Python 3.12+)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: ANYWIDGET_HMR=1


In [30]:
import numpy as np
import torch
import quantem.widget
from quantem.widget import Bin2D, profile
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
def make_haadf(size=512, seed=0):
    """Simulate HAADF-STEM image with atomic columns and scan noise."""
    rng = np.random.default_rng(seed)
    y, x = torch.meshgrid(
        torch.arange(size, device=device, dtype=torch.float32),
        torch.arange(size, device=device, dtype=torch.float32), indexing="ij")
    img = torch.zeros((size, size), device=device)
    spacing = 24.0
    sigma = 3.0
    for dy in range(-1, int(size / spacing) + 2):
        for dx in range(-1, int(size / spacing) + 2):
            cx = dx * spacing + (dy % 2) * spacing / 2
            cy = dy * spacing * 0.866
            r2 = (x - cx)**2 + (y - cy)**2
            img += torch.exp(-r2 / (2 * sigma**2))
    scan_noise = torch.from_numpy(rng.normal(0, 0.05, (size, size)).astype(np.float32)).to(device)
    img = img + scan_noise
    return img.cpu().numpy()
image = make_haadf(512, seed=42)
print(f"HAADF image: {image.shape}, device={device}")
print(f"quantem.widget {quantem.widget.__version__}")
profile()

HAADF image: (512, 512), device=mps
quantem.widget 0.0.12


In [31]:
# Side-by-side binning — 4× downsample with calibration
Bin2D(image, bin_factor=4, pixel_size=1.2, title="HAADF Atomic Columns")

Bin2D(512×512, bin=4×, binned=128×128, mode=mean, title='HAADF Atomic Columns')

In [32]:
# Get binned data for downstream analysis
w = Bin2D(image, bin_factor=8, pixel_size=1.2)
binned = w.get_binned_data()
print(f"Original: {image.shape} → Binned: {binned.shape}")
w.summary()

Original: (512, 512) → Binned: (64, 64)
Bin2D Summary
  Title: (none)
  Original: 512 × 512 (1 image)
  Bin factor: 8× (mean, crop)
  Binned: 64 × 64
  Pixel size: 1.2 Å/px → 9.6 Å/px
  Display: cmap=inferno, log_scale=False


In [33]:
# Load from file with IO — metadata auto-extracted
import tempfile, os
from quantem.widget import IO
tmp = tempfile.mkdtemp()
np.save(os.path.join(tmp, "sample.npy"), image)
result = IO.file(os.path.join(tmp, "sample.npy"))
Bin2D(result, bin_factor=4, title="Loaded via IO.file()")

Bin2D(512×512, bin=4×, binned=128×128, mode=mean, title='Loaded via IO.file()')